In [4]:
import os
import mlflow
import numpy as np
import torch
from transformers import ViTConfig, ViTModel, ViTImageProcessor, ViTForImageClassification
from torchvision import datasets
from torch.utils.data import DataLoader


mlflow.set_tracking_uri(f"sqlite:///{os.path.abspath('../../mlflow.db')}")

In [5]:
def custom_collate(examples):
    list_image = []
    list_label = []

    for image, label in examples:
        list_image.append(image)
        list_label.append(label)
    
    return list_image, torch.tensor(list_label)

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

test_raw = datasets.CIFAR10(root = '../../datasets', train = False, download = False)

model_id = 'nateraw/vit-base-patch16-224-cifar10'

processor = ViTImageProcessor.from_pretrained(model_id)
model = ViTForImageClassification.from_pretrained(model_id)

loader = DataLoader(test_raw, batch_size=32, collate_fn=custom_collate, shuffle=False)


c:\miniconda3\envs\aio\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

c:\miniconda3\envs\aio\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kiet0\.cache\huggingface\hub\models--nateraw--vit-base-patch16-224-cifar10. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/918 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/343M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

In [9]:
mlflow.set_experiment("CIFAR-10_Vit_CNN")

with mlflow.start_run(run_name="CNN"):
    mlflow.log_metric("test_accuracy", 0.7557)

In [8]:
samples = 0
correct = 0

model.to(device)

with mlflow.start_run(run_name="ViT"):
    with torch.no_grad():
        for images, labels in loader:
            inputs = processor(images=images, return_tensors="pt")
            labels = labels.to(device)
            inputs = inputs.to(device)

            outputs = model(**inputs)
            preds = outputs.logits.argmax(dim=1)

            correct += (preds == labels).sum().item()
            samples += labels.size(0)
    acc = correct/samples
    print(acc)
    mlflow.log_metric("test_accuracy", acc)



    

model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

0.9852
